# Kodagu Landslide Susceptibility: Random Forest, ANN and Map Agreement

This notebook prepares the extracted sample table, trains RF and ANN models, evaluates them, and provides optional raster prediction and existing-map analysis.

**Status:** repaired from an uploaded working notebook containing missing setup cells. Python syntax and dependencies between notebook variables have been reviewed. Full execution requires ArcGIS Pro, Spatial Analyst and the local input geodatabase; it has not been rerun against those inputs here.

## How to run

1. Open in the ArcGIS Pro Python environment containing scikit-learn, pandas, NumPy, Matplotlib and joblib. The uploaded notebook recorded scikit-learn 1.8.0.
2. Edit `PROJECT_ROOT` in the configuration cell. Confirm `Kodagu_ML_TrainingData` and the listed fields exist.
3. Restart the kernel and run from the beginning. RF and ANN modelling share the original 70:30 stratified split and random seed 42.
4. Leave the optional flags off for the initial model run. Enable raster mapping only after checking the input raster dictionary and rainfall layer identity.
5. Existing-map analysis requires the previously classified rasters and corrected Combine raster. Classification thresholds and the AHP model are **not reconstructed** in this notebook.

Generated files are directed to `Notebook_Repaired_Outputs`. Input datasets are read from the original geodatabase. Repeated runs may replace generated files in that new output directory.

### Previously saved results (historical, not a new execution)

| Model | Accuracy | Precision | Recall | F1 | ROC–AUC |
| --- | ---: | ---: | ---: | ---: | ---: |
| RF | 89.81% | 90.57% | 88.89% | 89.72% | 0.9702 |
| ANN | 81.48% | 80.36% | 83.33% | 81.82% | 0.9012 |

The uploaded outputs show **360 modelling records: 181 landslide and 179 background**, split into 252 training and 108 test records. This directly explains the test-set size. The reason an earlier sample dataset had 362 records is not documented here.

All executable outputs and execution counts have been cleared. The table above transcribes saved results; it does not claim the repaired raster procedure has reproduced the published maps.


## Configuration


In [ ]:
from pathlib import Path
import os

# Edit this one path to match the folder on your computer.
PROJECT_ROOT = Path(r"D:\Landslide_Project")
REPAIRED_ROOT = PROJECT_ROOT / "Notebook_Repaired_Outputs"
REPAIRED_ROOT.mkdir(parents=True, exist_ok=True)

# Start with model training and evaluation only. Enable mapping after checking inputs.
RUN_RASTER_MAPPING = False
RUN_EXISTING_MAP_ANALYSIS = False


## 1. Imports and environment


In [ ]:
# GIS processing
import arcpy
from arcpy.sa import ExtractMultiValuesToPoints
import os

# Data processing
import numpy as np
import pandas as pd

# Training and testing
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

# Data preprocessing
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Random Forest
from sklearn.ensemble import RandomForestClassifier

# Artificial Neural Network (MLP)
from sklearn.neural_network import MLPClassifier

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# Feature importance
from sklearn.inspection import permutation_importance

# Visualization
import matplotlib.pyplot as plt

# Save trained models
import joblib

print("All required libraries imported successfully!")


In [ ]:
arcpy.env.overwriteOutput = True
if RUN_RASTER_MAPPING or RUN_EXISTING_MAP_ANALYSIS:
    if arcpy.CheckExtension("Spatial") != "Available":
        raise RuntimeError("Spatial Analyst is required for the raster sections.")
    arcpy.CheckOutExtension("Spatial")
print("ArcGIS Pro imports are ready.")


In [ ]:
# Read integer/categorical rasters without inserting -9999 into an unsigned array.
# Keep the original numeric precision, then replace only masked NoData cells.
def read_numeric_raster(path, lower_left_corner=None, ncols=None, nrows=None,
                        nodata_to_value=-9999):
    raster = arcpy.Raster(path)
    args = (lower_left_corner, ncols, nrows)
    values = arcpy.RasterToNumPyArray(raster, *args, nodata_to_value=0).astype(np.float64)
    missing = arcpy.RasterToNumPyArray(arcpy.sa.IsNull(raster), *args, nodata_to_value=1)
    values[missing.astype(bool)] = nodata_to_value
    return values


## 2. Read the sample table


In [ ]:
import arcpy
import numpy as np 
import pandas as pd

arcpy.env.workspace = (str(PROJECT_ROOT / 'arcgis_files/Landslide/Landslide.gdb'))
fc = "Kodagu_ML_TrainingData"

fields = [f.name for f in arcpy.ListFields(fc)
         if f.type not in ("Geometry", "Blob", "Raster")]

records = []

with arcpy.da.SearchCursor(fc, fields) as cursor:
    for row in cursor:
        records.append(row)

df = pd.DataFrame(records, columns=fields)
df.head()


In [ ]:
print(df.isna().sum())


In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
data = df.copy()


In [ ]:
data.head()


In [ ]:
data = data.drop(columns = ['OBJECTID_1', 'gid', 'objectid', 'slide_no', 'state', 'district', 'toposheet_', 'slide_name', 'nh_sh_loca', 'latitude', 'longitude', 'length_in_', 'width_in_m', 'height_in_', 'area_in_sq', 'area_in__1', 'volume_in_', 'run_out_di', 'run_out__1', 'movement_t', 'movement_r', 'activity', 'distributi', 'style', 'failure_me', 'history', 'geomorphol', 'geology', 'structure', 'landuse_or', 'hydrologic', 'triggering', 'persons_de', 'people_aff', 'livestock_', 'communicat', 'infrastruc', 'landuse_af', 'geo_scient', 'preliminar', 'remarks', 'LULC_Kodagu1'])


In [ ]:
data.shape


In [ ]:
data.columns.tolist()


In [ ]:
data = data.drop(columns = ['CID'])


In [ ]:
data.head()


In [ ]:
data['LS_Label'].value_counts()


## 3. Check modelling fields and labels


In [ ]:
print("Dataset shape:", data.shape)

print("\nMissing values:")
print(data.isna().sum())

print("\nNoData values (-9999):")
print((data == -9999).sum())


In [ ]:
x = data[['Lith_Code','Geom_Code','Aspect','Curvature','Dist_Roads','Dist_Rivers_Kodagu','Elevation','Slope','LULC_Code','Rainfall','Dist_Lineaments','TWI']].copy()
y = data['LS_Label'].copy()

x.shape
y.shape


In [ ]:
print("y shape:", y.shape)


In [ ]:
x


In [ ]:
print(y.value_counts(dropna=False))


In [ ]:
from sklearn.model_selection import train_test_split


## 4. Shared train/test split


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.30, random_state=42,stratify=y)


In [ ]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)


## 5. Conditioning-factor types


In [ ]:
# Categorical conditioning factors
categorical_features = [
    "Lith_Code",
    "Geom_Code",
    "LULC_Code"
]

# Numerical conditioning factors
numerical_features = [
    "Aspect",
    "Curvature",
    "Dist_Roads",
    "Dist_Rivers_Kodagu",
    "Elevation",
    "Slope",
    "Rainfall",
    "Dist_Lineaments",
    "TWI"
]

print("Categorical factors:", len(categorical_features))
print("Numerical factors:", len(numerical_features))
print("Total factors:", len(categorical_features) + len(numerical_features))


## 6. Random Forest preprocessing and model


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Preprocess numerical factors
numerical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Preprocess categorical factors
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

# Combine both preprocessing steps
preprocessor = ColumnTransformer([
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing pipeline created successfully!")


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

print("Random Forest model created successfully!")


In [ ]:
rf_model.fit(x_train, y_train)

print("Random Forest training completed successfully!")


In [ ]:
print("Number of trees:",
      rf_model.named_steps["classifier"].n_estimators)

print("Classes:",
      rf_model.named_steps["classifier"].classes_)


In [ ]:
y_pred = rf_model.predict(x_test)

print("Predictions completed!")
print(y_pred[:10])


## 7. RF evaluation


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.metrics import roc_auc_score

y_prob = rf_model.predict_proba(x_test)[:, 1]

auc = roc_auc_score(y_test, y_prob)

print("Random Forest ROC-AUC:", round(auc, 4))


## 8. RF permutation importance and model export


In [ ]:
from sklearn.inspection import permutation_importance
import pandas as pd

result = permutation_importance(
    rf_model,
    x_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=1
)

importance_df = pd.DataFrame({
    "Factor": x_test.columns,
    "Importance": result.importances_mean,
    "Std_Dev": result.importances_std
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df.to_string(index=False))


In [ ]:
importance_df.to_csv(
    str(REPAIRED_ROOT / 'RF_Feature_Importance.csv'),
    index=False
)

print("Feature importance saved successfully!")


In [ ]:
import joblib
import os

output_folder = str(REPAIRED_ROOT / 'RF_Outputs')
os.makedirs(output_folder, exist_ok=True)

model_path = os.path.join(
    output_folder,
    "Kodagu_RF_Model.joblib"
)

joblib.dump(rf_model, model_path)

print("Random Forest model saved successfully!")
print(model_path)


In [ ]:
arcpy.env.workspace = str(PROJECT_ROOT / 'arcgis_files/Landslide/Landslide.gdb')
arcpy.ListRasters()


In [ ]:
data.columns


## 9. Optional raster preparation and RF mapping

Set `RUN_RASTER_MAPPING = True` only after verifying the input rasters. The original notebook used `kodaguannual_Rainfall` for mapping, while the project describes August–October 2018 rainfall. Confirm that this raster contains the same rainfall variable used to extract training values before generating a new map.

The deleted setup cell cannot be recovered exactly. The restored step aligns every factor to the DEM, uses nearest-neighbour resampling to retain existing values/category codes, and checks the resulting grid before prediction. Regenerated spatial outputs may differ from the historical maps.


In [ ]:
if RUN_RASTER_MAPPING:
    import arcpy
    import os

    gdb = str(PROJECT_ROOT / 'arcgis_files/Landslide/Landslide.gdb')

    raster_paths = {
        "Lith_Code": os.path.join(gdb, "RF_Align_Lith_Code"),
        "Geom_Code": os.path.join(gdb, "RF_Align_Geom_Code"),
        "Aspect": os.path.join(gdb, "Aspect_Kodag1"),
        "Curvature": os.path.join(gdb, "Curvatu_Koda1"),
        "Dist_Roads": os.path.join(gdb, "RF_Align_Dist_Roads"),
        "Dist_Rivers_Kodagu": os.path.join(gdb, "Dist_Rivers_Kodagu"),
        "Elevation": os.path.join(gdb, "KodaguDEM"),
        "Slope": os.path.join(gdb, "Slope_Kodagu2"),
        "LULC_Code": os.path.join(gdb, "RF_Align_LULC_Code"),
        "Rainfall": os.path.join(gdb, "kodaguannual_Rainfall"),
        "Dist_Lineaments": os.path.join(gdb, "RF_Align_Dist_Lineaments"),
        "TWI": os.path.join(gdb, "TWI_Kodagu_Filled")
    }


In [ ]:
if RUN_RASTER_MAPPING:
    for factor, path in raster_paths.items():
        if arcpy.Exists(path):
            print(f"{factor}: FOUND")
        else:
            print(f"{factor}: NOT FOUND")


In [ ]:
if RUN_RASTER_MAPPING:
    import pandas as pd

    raster_info = []

    for factor, path in raster_paths.items():
        if not arcpy.Exists(path):
            continue

        desc = arcpy.Describe(path)

        raster_info.append({
            "Factor": factor,
            "Cell_Size_X": round(desc.meanCellWidth, 4),
            "Cell_Size_Y": round(desc.meanCellHeight, 4),
            "Rows": desc.height,
            "Columns": desc.width,
            "XMin": round(desc.extent.XMin, 3),
            "YMax": round(desc.extent.YMax, 3),
            "Coordinate_System": desc.spatialReference.name
        })

    raster_info_df = pd.DataFrame(raster_info)

    print(raster_info_df.to_string(index=False))


In [ ]:
if RUN_RASTER_MAPPING:
    reference_raster = raster_paths["Elevation"]
    aligned_paths = raster_paths.copy()
    ref = arcpy.Describe(reference_raster)

    for factor, path in aligned_paths.items():

        d = arcpy.Describe(path)

        matched = (
            d.width == ref.width
            and d.height == ref.height
            and abs(d.meanCellWidth - ref.meanCellWidth) < 0.000001
            and abs(d.meanCellHeight - ref.meanCellHeight) < 0.000001
            and abs(d.extent.XMin - ref.extent.XMin) < 0.001
            and abs(d.extent.YMax - ref.extent.YMax) < 0.001
            and d.spatialReference.factoryCode ==
                ref.spatialReference.factoryCode
        )

        print(
            factor,
            "ALIGNED" if matched else "NOT ALIGNED",
            d.height,
            d.width
        )


In [ ]:
if RUN_RASTER_MAPPING:
    # Restored preparation step: construct a complete aligned predictor dictionary.
    # All input rasters must already use the DEM coordinate system.
    reference_raster = raster_paths["Elevation"]
    ref = arcpy.Describe(reference_raster)
    aligned_paths = {}
    final_paths = {}
    alignment_gdb = str(REPAIRED_ROOT / "Predictor_Alignment.gdb")
    if not arcpy.Exists(alignment_gdb):
        arcpy.management.CreateFileGDB(str(REPAIRED_ROOT), "Predictor_Alignment.gdb")

    rectangle = f"{ref.extent.XMin} {ref.extent.YMin} {ref.extent.XMax} {ref.extent.YMax}"

    # Nearest neighbour preserves category codes and existing sampled cell values.
    # These are reconstruction settings, not a recovery of the deleted original cell.
    with arcpy.EnvManager(snapRaster=reference_raster, cellSize=reference_raster,
                          outputCoordinateSystem=ref.spatialReference,
                          extent=ref.extent, resamplingMethod="NEAREST",
                          overwriteOutput=True):
        for factor, input_path in raster_paths.items():
            if not arcpy.Exists(input_path):
                raise FileNotFoundError(f"Missing predictor raster: {input_path}")
            d = arcpy.Describe(input_path)
            if d.spatialReference.exportToString() != ref.spatialReference.exportToString():
                raise ValueError(f"{factor}: project to the DEM coordinate system first.")
            if getattr(d, "bandCount", 1) != 1:
                raise ValueError(f"{factor}: expected a single-band predictor.")
            resampled_path = input_path
            if (abs(d.meanCellWidth-ref.meanCellWidth) > 1e-6 or
                    abs(d.meanCellHeight-ref.meanCellHeight) > 1e-6):
                resampled_path = os.path.join(alignment_gdb, "Resampled_" + factor)
                arcpy.management.Resample(input_path, resampled_path,
                                         f"{ref.meanCellWidth} {ref.meanCellHeight}", "NEAREST")
            aligned_paths[factor] = resampled_path
            output_path = os.path.join(alignment_gdb, "Final_" + factor)
            arcpy.management.Clip(
                in_raster=resampled_path, rectangle=rectangle, out_raster=output_path,
                in_template_dataset=reference_raster, clipping_geometry="NONE",
                maintain_clipping_extent="MAINTAIN_EXTENT")
            final_paths[factor] = output_path
            print(f"{factor}: prepared {output_path}")


In [ ]:
if RUN_RASTER_MAPPING:
    ref = arcpy.Describe(reference_raster)

    all_aligned = True

    for factor, path in final_paths.items():

        d = arcpy.Describe(path)

        aligned = (
            d.width == ref.width
            and d.height == ref.height
            and abs(d.meanCellWidth - ref.meanCellWidth) < 0.000001
            and abs(d.meanCellHeight - ref.meanCellHeight) < 0.000001
            and abs(d.extent.XMin - ref.extent.XMin) < 0.001
            and abs(d.extent.YMin - ref.extent.YMin) < 0.001
            and abs(d.extent.XMax - ref.extent.XMax) < 0.001
            and abs(d.extent.YMax - ref.extent.YMax) < 0.001
            and d.spatialReference.exportToString() ==
                ref.spatialReference.exportToString()
        )

        if not aligned:
            all_aligned = False

        print(
            factor,
            "ALIGNED" if aligned else "NOT ALIGNED",
            d.height,
            d.width
        )

    print("\nAll aligned:", all_aligned)
    if not all_aligned:
        raise ValueError("Raster grids differ; fix alignment before proceeding.")


In [ ]:
if RUN_RASTER_MAPPING:
    import numpy as np
    import pandas as pd

    if not all_aligned:
        raise ValueError("Correct raster alignment before proceeding.")

    nodata_report = []

    for factor, path in final_paths.items():

        arr = read_numeric_raster(
            path,
            nodata_to_value=-9999
        )

        valid = np.isfinite(arr) & (arr != -9999)

        nodata_report.append({
            "Factor": factor,
            "Valid_Cells": int(valid.sum()),
            "NoData_Cells": int((~valid).sum())
        })

        print(f"{factor}: checked")

        del arr, valid

    nodata_df = pd.DataFrame(nodata_report)

    print("\nRaster NoData report:")
    print(nodata_df.to_string(index=False))


In [ ]:
if RUN_RASTER_MAPPING:
    for factor, path in final_paths.items():
        print(f"{factor}: {path}")


In [ ]:
if RUN_RASTER_MAPPING:
    import arcpy
    from arcpy.sa import Raster, IsNull, Con, CellStatistics

    arcpy.CheckOutExtension("Spatial")

    # Reference raster
    reference_raster = final_paths["Elevation"]

    arcpy.env.snapRaster = reference_raster
    arcpy.env.cellSize = reference_raster
    arcpy.env.extent = arcpy.Describe(reference_raster).extent

    # Create a binary validity raster for each factor
    validity_rasters = []

    for factor, path in final_paths.items():

        r = Raster(path)

        valid_raster = Con(IsNull(r), 0, 1)

        validity_rasters.append(valid_raster)

        print(f"{factor}: validity checked")

    # Cell is 1 only when all 12 factors are valid
    common_valid = CellStatistics(
        validity_rasters,
        "MINIMUM",
        "DATA"
    )

    output_mask = str(REPAIRED_ROOT / 'RF_Outputs/RF_Common_Valid.tif')

    common_valid.save(output_mask)

    print("\nCommon valid-cell mask saved:")
    print(output_mask)


In [ ]:
if RUN_RASTER_MAPPING:
    import numpy as np

    mask_array = read_numeric_raster(output_mask)

    valid_count = np.count_nonzero(mask_array == 1)

    print("Cells with all 12 factors valid:", valid_count)


In [ ]:
if RUN_RASTER_MAPPING:
    # Reference raster
    reference_raster = final_paths["Elevation"]
    ref = arcpy.Describe(reference_raster)

    # Output location
    output_folder = str(REPAIRED_ROOT / 'RF_Outputs')
    os.makedirs(output_folder, exist_ok=True)

    output_path = os.path.join(
        output_folder,
        "Kodagu_RF_Susceptibility.tif"
    )

    # Exact predictor names used during RF training
    predictor_columns = [
        "Lith_Code",
        "Geom_Code",
        "Aspect",
        "Curvature",
        "Dist_Roads",
        "Dist_Rivers_Kodagu",
        "Elevation",
        "Slope",
        "LULC_Code",
        "Rainfall",
        "Dist_Lineaments",
        "TWI"
    ]

    # Rows processed per batch
    batch_rows = 128

    # NoData marker
    nodata = -9999.0

    print("Rows:", ref.height)
    print("Columns:", ref.width)
    print("Factors:", len(predictor_columns))
    print("Output:", output_path)


In [ ]:
if RUN_RASTER_MAPPING:
    # Check alignment before prediction
    for factor in predictor_columns:
        d = arcpy.Describe(final_paths[factor])

        if (
            d.width != ref.width
            or d.height != ref.height
            or abs(d.extent.XMin - ref.extent.XMin) > 0.001
            or abs(d.extent.YMax - ref.extent.YMax) > 0.001
            or abs(d.meanCellWidth - ref.meanCellWidth) > 0.000001
            or abs(d.meanCellHeight - ref.meanCellHeight) > 0.000001
            or d.spatialReference.factoryCode != ref.spatialReference.factoryCode
        ):
            raise ValueError(f"{factor} is not aligned!")

    # Allocate output array
    prediction_array = np.full(
        (ref.height, ref.width),
        nodata,
        dtype=np.float32
    )

    # Identify the probability column for class 1
    class_index = list(rf_model.classes_).index(1)

    # Process raster rows in batches
    for start_row in range(0, ref.height, batch_rows):

        nrows = min(batch_rows, ref.height - start_row)

        # Lower-left corner of this batch
        lower_left = arcpy.Point(
            ref.extent.XMin,
            ref.extent.YMax -
            (start_row + nrows) * ref.meanCellHeight
        )

        block_data = {}

        valid_mask = np.ones(
            (nrows, ref.width),
            dtype=bool
        )

        for factor in predictor_columns:

            arr = read_numeric_raster(
                final_paths[factor],
                lower_left,
                ref.width,
                nrows,
                nodata_to_value=nodata
            )

            arr = arr.astype(np.float64)

            valid_mask &= (
                np.isfinite(arr) & (arr != nodata)
            )

            block_data[factor] = arr.ravel()

        # Skip blocks without valid cells
        if not valid_mask.any():
            continue

        # Construct model input
        block_df = pd.DataFrame(
            block_data,
            columns=predictor_columns
        )

        flat_valid = valid_mask.ravel()

        # Predict class-1 probability
        probabilities = rf_model.predict_proba(
            block_df.loc[flat_valid]
        )[:, class_index]

        # Store predictions
        block_output = np.full(
            nrows * ref.width,
            nodata,
            dtype=np.float32
        )

        block_output[flat_valid] = probabilities

        prediction_array[
            start_row:start_row + nrows, :
        ] = block_output.reshape(nrows, ref.width)

        print(
            f"Processed rows "
            f"{start_row + 1} to "
            f"{start_row + nrows}"
        )

    print("Random Forest prediction completed!")


In [ ]:
if RUN_RASTER_MAPPING:
    # Convert NumPy array to raster
    lower_left = arcpy.Point(
        ref.extent.XMin,
        ref.extent.YMin
    )

    rf_raster = arcpy.NumPyArrayToRaster(
        prediction_array,
        lower_left,
        ref.meanCellWidth,
        ref.meanCellHeight,
        value_to_nodata=nodata
    )

    # Save output
    rf_raster.save(output_path)

    # Set coordinate system
    arcpy.management.DefineProjection(
        output_path,
        ref.spatialReference
    )

    print("RF susceptibility raster saved!")
    print(output_path)


In [ ]:
if RUN_RASTER_MAPPING:
    valid_predictions = prediction_array[
        prediction_array != nodata
    ]

    if not len(valid_predictions):
        raise ValueError("No valid RF prediction cells; check the predictor masks.")
    print("Valid prediction cells:", len(valid_predictions))
    print("Minimum susceptibility:", valid_predictions.min())
    print("Maximum susceptibility:", valid_predictions.max())
    print("Mean susceptibility:", valid_predictions.mean())


## 10. Optional statistics for the existing RF class map

Set `RUN_EXISTING_MAP_ANALYSIS = True` to summarise existing classified maps. Three-class rasters must already exist, with 1 = Low, 2 = Moderate and 3 = High. This section does not create the original class thresholds.


In [ ]:
if RUN_EXISTING_MAP_ANALYSIS:
    import arcpy
    import pandas as pd

    rf_class_raster = (
        str(PROJECT_ROOT / 'arcgis_files/Landslide/Landslide.gdb/Kodagu_RF_Susceptibility_3Class')
    )

    # Build raster attribute table if needed
    arcpy.management.BuildRasterAttributeTable(
        rf_class_raster, "Overwrite"
    )

    # Extract class counts
    records = []

    with arcpy.da.SearchCursor(
        rf_class_raster, ["Value", "Count"]
    ) as cursor:
        for value, count in cursor:
            records.append((value, count))

    results = pd.DataFrame(
        records, columns=["Class", "Cell_Count"]
    )

    class_names = {
        1: "Low",
        2: "Moderate",
        3: "High"
    }

    results["Susceptibility"] = results["Class"].map(class_names)

    results["Area_Percent"] = (
        results["Cell_Count"] / results["Cell_Count"].sum() * 100
    )

    print(results.to_string(index=False))


## 11. ANN preprocessing and model


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

print("ANN libraries imported successfully!")


In [ ]:
ann_num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Numerical preprocessing created!")


In [ ]:
ann_cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

print("Categorical preprocessing created!")


In [ ]:
ann_preprocessor = ColumnTransformer([
    ("num", ann_num_transformer, numerical_features),
    ("cat", ann_cat_transformer, categorical_features)
])

print("ANN preprocessing pipeline created!")


In [ ]:
ann_model = Pipeline([
    ("preprocessing", ann_preprocessor),
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=0.01,
        learning_rate_init=0.001,
        max_iter=1000,
        early_stopping=True,
        random_state=42
    ))
])

print("ANN model created successfully!")


In [ ]:
print(ann_model)


In [ ]:
ann_model.fit(x_train, y_train)

print("ANN training completed successfully!")


In [ ]:
ann_classifier = ann_model.named_steps["classifier"]

print("Training iterations:", ann_classifier.n_iter_)
print("Final training loss:", ann_classifier.loss_)
print("Hidden layers:", ann_classifier.hidden_layer_sizes)

print("Training stopped:", ann_classifier.n_iter_ < ann_classifier.max_iter)


## 12. ANN evaluation and export


In [ ]:
ann_y_pred = ann_model.predict(x_test)

ann_y_prob = ann_model.predict_proba(x_test)[:, 1]

print("First 10 predicted classes:")
print(ann_y_pred[:10])

print("\nFirst 10 predicted probabilities:")
print(ann_y_prob[:10])


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    classification_report
)

print("ANN Accuracy:",
      accuracy_score(y_test, ann_y_pred))

print("ANN Precision:",
      precision_score(y_test, ann_y_pred))

print("ANN Recall:",
      recall_score(y_test, ann_y_pred))

print("ANN F1-score:",
      f1_score(y_test, ann_y_pred))

print("ANN ROC-AUC:",
      roc_auc_score(y_test, ann_y_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, ann_y_pred))

print("\nClassification Report:")
print(classification_report(y_test, ann_y_pred))


In [ ]:
import joblib
import os

output_folder = str(REPAIRED_ROOT / 'ANN_Outputs')

os.makedirs(output_folder, exist_ok=True)

ann_model_path = os.path.join(
    output_folder,
    "Kodagu_ANN_Model.joblib"
)

joblib.dump(ann_model, ann_model_path)

print("ANN model saved successfully!")
print(ann_model_path)


## 13. Optional ANN raster prediction


In [ ]:
if RUN_RASTER_MAPPING:
    import arcpy
    import numpy as np
    import pandas as pd
    import os

    # Output folder
    ann_output_folder = str(REPAIRED_ROOT / 'ANN_Outputs')
    os.makedirs(ann_output_folder, exist_ok=True)

    # Output susceptibility raster
    ann_output_path = os.path.join(
        ann_output_folder,
        "Kodagu_ANN_Susceptibility.tif"
    )

    # Reference raster
    ann_reference = final_paths["Elevation"]
    ann_ref = arcpy.Describe(ann_reference)

    # Predictor columns in training order
    ann_predictors = [
        "Lith_Code",
        "Geom_Code",
        "Aspect",
        "Curvature",
        "Dist_Roads",
        "Dist_Rivers_Kodagu",
        "Elevation",
        "Slope",
        "LULC_Code",
        "Rainfall",
        "Dist_Lineaments",
        "TWI"
    ]

    ann_nodata = -9999.0
    ann_batch_rows = 64

    print("Raster rows:", ann_ref.height)
    print("Raster columns:", ann_ref.width)
    print("Number of factors:", len(ann_predictors))


In [ ]:
if RUN_RASTER_MAPPING:
    # Confirm alignment before making predictions
    for factor in ann_predictors:
        d = arcpy.Describe(final_paths[factor])

        if (
            d.width != ann_ref.width
            or d.height != ann_ref.height
            or abs(d.extent.XMin - ann_ref.extent.XMin) > 0.001
            or abs(d.extent.YMin - ann_ref.extent.YMin) > 0.001
            or abs(d.extent.XMax - ann_ref.extent.XMax) > 0.001
            or abs(d.extent.YMax - ann_ref.extent.YMax) > 0.001
        ):
            raise ValueError(f"{factor} is not aligned!")

    # Empty output array
    ann_prediction_array = np.full(
        (ann_ref.height, ann_ref.width),
        ann_nodata,
        dtype=np.float32
    )

    # Identify landslide class probability column
    ann_class_index = list(ann_model.classes_).index(1)

    # Process the rows in blocks
    for start_row in range(0, ann_ref.height, ann_batch_rows):

        nrows = min(
            ann_batch_rows,
            ann_ref.height - start_row
        )

        # Lower-left corner of current block
        lower_left = arcpy.Point(
            ann_ref.extent.XMin,
            ann_ref.extent.YMax -
            (start_row + nrows) * ann_ref.meanCellHeight
        )

        block_data = {}

        valid_mask = np.ones(
            (nrows, ann_ref.width),
            dtype=bool
        )

        for factor in ann_predictors:

            arr = read_numeric_raster(
                final_paths[factor],
                lower_left,
                ann_ref.width,
                nrows,
                nodata_to_value=ann_nodata
            )

            arr = arr.astype(np.float64)

            valid_mask &= (
                np.isfinite(arr) &
                (arr != ann_nodata)
            )

            block_data[factor] = arr.ravel()

        if not valid_mask.any():
            continue

        block_df = pd.DataFrame(
            block_data,
            columns=ann_predictors
        )

        flat_valid = valid_mask.ravel()

        # ANN class-1 probability
        probabilities = ann_model.predict_proba(
            block_df.loc[flat_valid]
        )[:, ann_class_index]

        block_output = np.full(
            nrows * ann_ref.width,
            ann_nodata,
            dtype=np.float32
        )

        block_output[flat_valid] = probabilities

        ann_prediction_array[
            start_row:start_row + nrows, :
        ] = block_output.reshape(
            nrows,
            ann_ref.width
        )

        print(
            f"Processed rows "
            f"{start_row + 1} to "
            f"{start_row + nrows}"
        )

    print("ANN raster prediction completed!")


In [ ]:
if RUN_RASTER_MAPPING:
    ann_lower_left = arcpy.Point(
        ann_ref.extent.XMin,
        ann_ref.extent.YMin
    )

    ann_raster = arcpy.NumPyArrayToRaster(
        ann_prediction_array,
        ann_lower_left,
        ann_ref.meanCellWidth,
        ann_ref.meanCellHeight,
        value_to_nodata=ann_nodata
    )

    ann_raster.save(ann_output_path)

    arcpy.management.DefineProjection(
        ann_output_path,
        ann_ref.spatialReference
    )

    print("ANN susceptibility raster saved!")
    print(ann_output_path)


In [ ]:
if RUN_RASTER_MAPPING:
    ann_valid = ann_prediction_array[
        ann_prediction_array != ann_nodata
    ]

    if not len(ann_valid):
        raise ValueError("No valid ANN prediction cells; check the predictor masks.")
    print("Valid prediction cells:", len(ann_valid))
    print("Minimum susceptibility:", ann_valid.min())
    print("Maximum susceptibility:", ann_valid.max())
    print("Mean susceptibility:", ann_valid.mean())


## 14. Optional existing ANN class-map statistics


In [ ]:
if RUN_EXISTING_MAP_ANALYSIS:
    import arcpy
    import pandas as pd

    ann_class_raster = (
        str(PROJECT_ROOT / 'arcgis_files/Landslide/Landslide.gdb/Kodagu_ANN')
    )

    arcpy.management.BuildRasterAttributeTable(
        ann_class_raster,
        "Overwrite"
    )

    records = []

    with arcpy.da.SearchCursor(
        ann_class_raster,
        ["Value", "Count"]
    ) as cursor:
        for value, count in cursor:
            records.append((int(value), int(count)))

    ann_results = pd.DataFrame(
        records,
        columns=["Class", "Cell_Count"]
    )

    class_names = {
        1: "Low",
        2: "Moderate",
        3: "High"
    }

    ann_results["Susceptibility"] = (
        ann_results["Class"].map(class_names)
    )

    ann_results["Area_Percent"] = (
        ann_results["Cell_Count"] /
        ann_results["Cell_Count"].sum() * 100
    )

    print(ann_results.to_string(index=False))


## 15. Optional existing-map agreement analysis

A working copy of the Combine raster is created before adding agreement codes; the original Combine dataset is retained.


In [ ]:
if RUN_EXISTING_MAP_ANALYSIS:
    gdb = str(PROJECT_ROOT / "arcgis_files/Landslide/Landslide.gdb")
    source_combine = os.path.join(gdb, "Kodagu_ThreeModel_Combine_Corrected")
    analysis_gdb = str(REPAIRED_ROOT / "Map_Analysis.gdb")
    if not arcpy.Exists(analysis_gdb):
        arcpy.management.CreateFileGDB(str(REPAIRED_ROOT), "Map_Analysis.gdb")
    combine_raster = os.path.join(analysis_gdb, "ThreeModel_Combine")
    arcpy.management.CopyRaster(source_combine, combine_raster)
    fields = [f.name for f in arcpy.ListFields(combine_raster)]
    print("Available fields:", fields)


In [ ]:
if RUN_EXISTING_MAP_ANALYSIS:
    # Replace with actual Combine attribute table field names
    ahp_field = "Kodagu_AHP_3Class_Corrected"
    rf_field = "Kodagu_RF_Susceptibility_3Class"
    ann_field = "Kodagu_ANN"

    records = []

    with arcpy.da.SearchCursor(
        combine_raster,
        [ahp_field, rf_field, ann_field, "Count"]
    ) as cursor:

        for ahp, rf, ann, count in cursor:

            if ahp == rf == ann:
                agreement = "All three agree"

            elif ahp == rf or ahp == ann or rf == ann:
                agreement = "Two agree"

            else:
                agreement = "All three disagree"

            records.append({
                "Agreement": agreement,
                "Cell_Count": int(count)
            })

    agreement_df = pd.DataFrame(records)

    agreement_summary = (
        agreement_df.groupby("Agreement")["Cell_Count"]
        .sum()
        .reset_index()
    )

    agreement_summary["Percentage"] = (
        agreement_summary["Cell_Count"]
        / agreement_summary["Cell_Count"].sum()
        * 100
    )

    print(agreement_summary.to_string(index=False))


In [ ]:
if RUN_EXISTING_MAP_ANALYSIS:
    import arcpy

    combine_raster = (
        os.path.join(analysis_gdb, "ThreeModel_Combine")
    )

    # Use the ahp_field, rf_field, ann_field
    # variables defined in your previous step.

    field_names = [f.name for f in arcpy.ListFields(combine_raster)]

    if "Agreement_Code" not in field_names:
        arcpy.management.AddField(
            combine_raster,
            "Agreement_Code",
            "SHORT"
        )

    with arcpy.da.UpdateCursor(
        combine_raster,
        [ahp_field, rf_field, ann_field, "Agreement_Code"]
    ) as cursor:

        for row in cursor:

            ahp, rf, ann = row[:3]

            if ahp == rf == ann:
                row[3] = 3

            elif ahp == rf or ahp == ann or rf == ann:
                row[3] = 2

            else:
                row[3] = 1

            cursor.updateRow(row)

    print("Agreement codes assigned successfully!")


In [ ]:
if RUN_EXISTING_MAP_ANALYSIS:
    from arcpy.sa import Lookup

    agreement_raster = Lookup(
        combine_raster,
        "Agreement_Code"
    )

    agreement_output = (
        str(REPAIRED_ROOT / "Kodagu_Model_Agreement.tif")
    )

    agreement_raster.save(agreement_output)

    print("Spatial agreement raster created successfully!")


In [ ]:
if RUN_EXISTING_MAP_ANALYSIS:
    import arcpy
    from arcpy.sa import Raster, Lookup, Con

    arcpy.CheckOutExtension("Spatial")

    combine_raster = (
        os.path.join(analysis_gdb, "ThreeModel_Combine")
    )

    # Retrieve each model's class values
    ahp = Lookup(combine_raster, ahp_field)
    rf = Lookup(combine_raster, rf_field)
    ann = Lookup(combine_raster, ann_field)

    # 1 = all three models identify high susceptibility
    # 0 = other valid class combinations
    high_agreement = Con(
        (ahp == 3) & (rf == 3) & (ann == 3),
        1,
        0
    )

    output_high = (
        str(REPAIRED_ROOT / "Kodagu_High_Susceptibility_Agreement.tif")
    )

    high_agreement.save(output_high)

    print("High-susceptibility agreement raster created!")


In [ ]:
if RUN_EXISTING_MAP_ANALYSIS:
    import arcpy

    output_high = (
        str(REPAIRED_ROOT / "Kodagu_High_Susceptibility_Agreement.tif")
    )

    arcpy.management.BuildRasterAttributeTable(output_high, "Overwrite")
    counts = {}

    with arcpy.da.SearchCursor(
        output_high, ["Value", "Count"]
    ) as cursor:
        for value, count in cursor:
            counts[int(value)] = int(count)

    high_cells = counts.get(1, 0)
    total_cells = sum(counts.values())

    desc = arcpy.Describe(output_high)

    cell_area_km2 = (
        desc.meanCellWidth * desc.meanCellHeight
    ) / 1_000_000

    high_area = high_cells * cell_area_km2
    high_percentage = high_cells / total_cells * 100

    print("High-agreement cells:", high_cells)
    print("High-agreement area (km²):", round(high_area, 2))
    print("High-agreement percentage:", round(high_percentage, 2))


## Repair and interpretation notes

- The original first failure used `reference_raster` before assignment; `aligned_paths` was absent from the notebook. Subsequent undefined names were cascading failures.
- The raster alignment block is a replacement implementation, not recovered historical source code. It deliberately stops if CRS or final grid checks fail.
- Original RF/ANN model parameters, predictor lists and split settings are retained. Aspect remains a numerical degree-valued feature to preserve the original model design.
- Sample metadata fields with missing values are removed before modelling; missing inventory descriptions for background records are not themselves missing predictors.
- RF/ANN classification, AHP weighting/ROC and creation of the initial Combine raster were performed outside the available code. Supply those existing datasets for the optional analysis.
- Background-sampling restrictions and a random split can affect apparent generalisation. Spatial agreement is not independent validation.
- Raw rasters and the training geodatabase are not bundled with this notebook.

### ArcGIS references used for the repair

- [Snap Raster](https://pro.arcgis.com/en/pro-app/latest/tool-reference/environment-settings/snap-raster.htm)
- [Resample](https://pro.arcgis.com/en/pro-app/latest/tool-reference/data-management/resample.htm)
- [Clip Raster](https://pro.arcgis.com/en/pro-app/latest/tool-reference/data-management/clip.htm)
- [RasterToNumPyArray](https://pro.arcgis.com/en/pro-app/latest/arcpy/functions/rastertonumpyarray-function.htm)
